# US Census Designated Places and Demographic Data Processing

**Overview:**

This workbook automates the process of downloading, extracting, and processing US Census designated places shapefiles alongside associated demographic data. The workflow is structured into the following key steps:

1. **Download Shapefiles:**  
   For each specified state (identified by its FIPS code), the 2024 Census designated places shapefile is downloaded from the US Census TIGER/Line repository.

2. **Extract Shapefiles:**  
   The downloaded ZIP files are extracted into state-specific directories for further processing.

3. **Download Census Data:**  
   Demographic data is fetched from the 2023 ACS 5-year dataset via the Census API using a valid Census API key.

4. **Join Data:**  
   The downloaded demographic data is merged with the corresponding shapefile data based on common FIPS codes (state and place identifiers).

5. **Filter Data:**  
   The merged dataset is filtered to retain only those places with a Total Population of less than 50,000.

6. **Save Output:**  
   The final processed shapefiles and demographic CSV files are saved to a designated output directory.

**Key Features:**

- **Parallel Processing:**  
  Utilizes `ProcessPoolExecutor` to concurrently process multiple states, significantly reducing overall runtime.

- **Robust Data Retrieval:**  
  Implements retry logic with exponential backoff (via the `tenacity` library) to handle transient network issues during data fetching.

- **Logging:**  
  Detailed logging is integrated throughout the process to track progress and troubleshoot errors effectively.

- **Descriptive Variable Mapping:**  
  ACS variable codes are mapped to descriptive names for improved clarity and easier interpretation of demographic data.

**Data Sources:**

- **US Census Shapefiles:**  
  [2024 TIGER/Line Shapefiles for Designated Places](https://www2.census.gov/geo/tiger/TIGER2024/PLACE/)

- **Census Demographic Data (ACS 2023):**  
  [US Census ACS 5-Year Data](https://api.census.gov/data/2023/acs/acs5)

**Configuration Notes:**

- **Census API Key:**  
  Ensure that a valid Census API key is set in the `CENSUS_API_KEY` environment variable before running the script.

- **Output Directory:**  
  Processed files will be saved in a pre-configured output directory, which is created if it does not already exist.

In [1]:
import os
import requests
from zipfile import ZipFile
from concurrent.futures import ProcessPoolExecutor, as_completed
import glob
import json
import logging
import csv
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

########################################
#             CONFIGURATION
########################################

# Configure Logging
logging.basicConfig(
    level=logging.INFO,  # Set to INFO to reduce verbosity
    format='[%(levelname)s] %(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Census API Key (Using Dummy Key as per User's Request)
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY", "379fc806fcab089c8c094ba37b9cc01b2f1e76bf")
if not CENSUS_API_KEY:
    logger.error("CENSUS_API_KEY is not set. Please set it before running the script.")
    exit(1)

# ACS endpoint example (2023 5-year ACS)
census_data_base_url = "https://api.census.gov/data/2023/acs/acs5"

# Base URL for the 2024 places shapefiles
base_url = "https://www2.census.gov/geo/tiger/TIGER2024/PLACE/"
logger.info("Base URL set for shapefile downloads.")

# Updated Directory to Save Outputs - Mona June 4, 2025
output_dir = "/home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/us_census_designated_places/2024"
os.makedirs(output_dir, exist_ok=True)
logger.info(f"Output directory ensured at: {output_dir}")

# State FIPS codes and names
state_fips_names = {
    "02": "Alaska",
    "04": "Arizona",
    "06": "California",
    "08": "Colorado",
    "16": "Idaho",
    "30": "Montana",
    "32": "Nevada",
    "35": "New_Mexico",
    "41": "Oregon",
    "49": "Utah",
    "53": "Washington",
    "56": "Wyoming"
}
logger.info(f"Total states to process: {len(state_fips_names)}")

########################################
#             HELPER FUNCTIONS
########################################

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True
)
def fetch_url(url, params=None, timeout=120):
    print("Fetching URL:", url)
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response

def download_and_extract(fips_code, state_name):
    ########################################
    #        DOWNLOAD AND EXTRACT
    ########################################
    print("Starting download_and_extract for", state_name)
    logger.info(f"Starting download for {state_name} (FIPS code {fips_code})...")
    url = f"{base_url}tl_2024_{fips_code}_place.zip"
    try:
        response = fetch_url(url)
        logger.info(f"Downloading shapefile for {state_name} (FIPS code {fips_code}) from {url}")
        
        state_dir = os.path.join(output_dir, f"{state_name}_{fips_code}")
        os.makedirs(state_dir, exist_ok=True)
        logger.info(f"Directory created for {state_name}: {state_dir}")

        zip_path = os.path.join(state_dir, f"tl_2024_{fips_code}_place.zip")
        with open(zip_path, "wb") as f:
            print("Writing ZIP for", state_name)
            f.write(response.content)
        logger.info(f"ZIP file saved for {state_name} at {zip_path}")

        with ZipFile(zip_path, 'r') as z:
            print("Extracting ZIP for", state_name)
            z.extractall(state_dir)
        logger.info(f"ZIP file extracted for {state_name}.")

        extracted_files = glob.glob(os.path.join(state_dir, "*"))
        logger.info(f"Extracted {len(extracted_files)} files for {state_name} to {state_dir}")
        if len(extracted_files) == 0:
            logger.warning(f"No files extracted for {state_name}. Check if the ZIP was empty or malformed.")

        ########################################
        #        DOWNLOAD CENSUS DATA
        ########################################
        download_census_data_for_state(fips_code, state_name, state_dir)

        ########################################
        #        JOIN DEMOGRAPHIC DATA
        ########################################
        join_demographics_with_shapefile(state_fips=fips_code, state_name=state_name, state_dir=state_dir)

    except requests.exceptions.HTTPError as http_err:
        logger.error(f"HTTP error occurred while downloading shapefile for {state_name} (FIPS {fips_code}): {http_err}")
    except requests.exceptions.RequestException as req_err:
        logger.error(f"Request exception for {state_name} (FIPS {fips_code}): {req_err}")
    except Exception as e:
        logger.error(f"Exception occurred while processing {state_name} (FIPS {fips_code}): {e}")

def join_demographics_with_shapefile(state_fips, state_name, state_dir):
    ########################################
    #        JOIN DEMOGRAPHICS WITH SHP
    ########################################
    print("Starting join_demographics_with_shapefile for", state_name)
    try:
        shapefile_pattern = os.path.join(state_dir, "tl_2024_*_place.shp")
        shapefile_paths = glob.glob(shapefile_pattern)
        if not shapefile_paths:
            logger.error(f"No shapefile found in {state_dir} matching pattern {shapefile_pattern}")
            return
        shapefile_path = shapefile_paths[0]
        logger.info(f"Shapefile found: {shapefile_path}")

        # Read shapefile
        print("Reading shapefile for", state_name)
        gdf = gpd.read_file(shapefile_path)
        logger.info(f"Shapefile read into GeoDataFrame with {len(gdf)} records.")

        # Read CSV
        csv_path = os.path.join(state_dir, f"{state_name}_{state_fips}_demographics.csv")
        if not os.path.exists(csv_path):
            logger.error(f"CSV file not found: {csv_path}")
            return
        print("Reading CSV for", state_name)
        df = pd.read_csv(csv_path)
        logger.info(f"CSV file read into DataFrame with {len(df)} records.")

        # Zero-pad FIPS
        print("Adjusting FIPS codes for join", state_name)
        gdf['State FIPS'] = gdf['STATEFP'].astype(str).str.zfill(2)
        gdf['Place FIPS'] = gdf['PLACEFP'].astype(str).str.zfill(5)
        df['State FIPS'] = df['State FIPS'].astype(str).str.zfill(2)
        df['Place FIPS'] = df['Place FIPS'].astype(str).str.zfill(5)

        # Create join key
        gdf['GEOID_join'] = gdf['State FIPS'] + gdf['Place FIPS']
        df['GEOID_join'] = df['State FIPS'] + df['Place FIPS']

        # Identify demographic columns
        demographic_columns = [c for c in df.columns if c in header_mapping.values() or c == "GEOID_join"]

        # Merge
        print("Merging data for", state_name)
        merged_gdf = gdf.merge(df[demographic_columns], on='GEOID_join', how='left')
        logger.info(f"GeoDataFrame after merge has {len(merged_gdf)} records.")

        # Check missing
        missing_demographics = merged_gdf['Total Population'].isnull().sum()
        if missing_demographics > 0:
            logger.warning(f"{missing_demographics} records have missing demographic data.")

        # Filter to only keep places that have < 50,000 Total Population
        print("Filtering CDPs with Total Population < 50000 for", state_name)
        merged_gdf['Total Population'] = pd.to_numeric(merged_gdf['Total Population'], errors='coerce')
        merged_gdf = merged_gdf[merged_gdf['Total Population'] < 50000]

        # Drop join key
        if 'GEOID_join' in merged_gdf.columns:
            merged_gdf = merged_gdf.drop(columns=['GEOID_join'])

        # Save updated shapefile
        output_shapefile_path = os.path.join(state_dir, f"tl_2024_{state_fips}_place.shp")
        print("Saving updated shapefile for", state_name)
        merged_gdf.to_file(output_shapefile_path)
        logger.info(f"Updated shapefile with demographics saved at {output_shapefile_path}")

    except Exception as e:
        logger.error(f"Exception occurred while joining demographics with shapefile for {state_name} (FIPS {state_fips}): {e}")

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True
)
def download_census_data_for_state(state_fips, state_name, state_dir):
    ########################################
    #        DOWNLOAD CENSUS DATA
    ########################################
    print("Starting download_census_data_for_state for", state_name)
    logger.info(f"Fetching Census data for {state_name} (FIPS {state_fips})...")

    var_str = ",".join(["NAME"])
    params = {
        "get": var_str,
        "for": "place:*",
        "in": f"state:{state_fips}",
        "key": CENSUS_API_KEY
    }

    try:
        response = fetch_url(census_data_base_url, params=params)
        print("Parsing JSON for", state_name)
        try:
            data = response.json()
            logger.info(f"JSON data successfully parsed for {state_name} (FIPS {state_fips}).")
        except json.JSONDecodeError as json_err:
            logger.error(f"JSON decode error for {state_name} (FIPS {state_fips}): {json_err}")
            logger.debug(f"Response content: {response.text}")
            return

        headers = data[0]
        rows = data[1:]

        # Map headers
        mapped_headers = [header_mapping.get(header, header) for header in headers]

        # Prepare data for DataFrame
        csv_data = [mapped_headers]
        for row in rows:
            row_dict = dict(zip(headers, row))
            mapped_row = {header_mapping.get(k, k): v for k, v in row_dict.items()}
            csv_data.append(mapped_row)

        output_csv_path = os.path.join(state_dir, f"{state_name}_{state_fips}_demographics.csv")

        print("Saving CSV for", state_name)
        df = pd.DataFrame(csv_data[1:], columns=csv_data[0])
        df.to_csv(output_csv_path, index=False)
        logger.info(f"Saved demographic data for {state_name} to {output_csv_path}")

    except requests.exceptions.HTTPError as http_err:
        logger.error(f"HTTP error occurred while fetching Census data for {state_name} (FIPS {state_fips}): {http_err}")
        logger.debug(f"Response content: {response.text}")
    except requests.exceptions.RequestException as req_err:
        logger.error(f"Request exception for Census data for {state_name} (FIPS {state_fips}): {req_err}")
    except Exception as e:
        logger.error(f"Exception occurred while retrieving Census data for {state_name} (FIPS {state_fips}): {e}")

########################################
#             MAIN PROCESS
########################################

print("Calling main process now.")
logger.info("Preparing to start parallel processing.")
tasks = [(fips_code, state_name) for fips_code, state_name in state_fips_names.items()]
logger.info("Submitting tasks to the executor.")

with ProcessPoolExecutor(max_workers=10) as executor:
    future_to_task = {executor.submit(download_and_extract, fips, name): (fips, name) for fips, name in tasks}
    for future in tqdm(as_completed(future_to_task), total=len(future_to_task), desc="Processing States"):
        fips, name = future_to_task[future]
        try:
            future.result()
            logger.info(f"Completed task for {name} (FIPS code {fips})")
        except Exception as exc:
            logger.error(f"{name} (FIPS code {fips}) generated an exception: {exc}")

logger.info("All tasks have been processed. Check the output directory for the extracted state folders and demographic CSV files.")


[INFO] 2025-07-03 16:11:06 - Base URL set for shapefile downloads.
[INFO] 2025-07-03 16:11:06 - Output directory ensured at: /home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/us_census_designated_places/2024
[INFO] 2025-07-03 16:11:06 - Total states to process: 12
[INFO] 2025-07-03 16:11:06 - Preparing to start parallel processing.
[INFO] 2025-07-03 16:11:06 - Submitting tasks to the executor.


Calling main process now.


Processing States:   0%|          | 0/12 [00:00<?, ?it/s]

Starting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract forStarting download_and_extract for         AlaskaArizonaColorado IdahoUtahCaliforniaOregonMontana
Nevada

New_Mexico







[INFO] 2025-07-03 16:11:07 - Starting download for Alaska (FIPS code 02)...


...

[INFO] 2025-07-03 16:11:09 - JSON data successfully parsed for Wyoming (FIPS 56).
[ERROR] 2025-07-03 16:11:09 - Exception occurred while retrieving Census data for Wyoming (FIPS 56): name 'header_mapping' is not defined


Starting join_demographics_with_shapefile for Wyoming


[INFO] 2025-07-03 16:11:09 - Shapefile found: /home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/us_census_designated_places/2024/Wyoming_56/tl_2024_56_place.shp


Reading shapefile for Wyoming


[INFO] 2025-07-03 16:11:09 - Shapefile read into GeoDataFrame with 205 records.
[ERROR] 2025-07-03 16:11:09 - CSV file not found: /home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/us_census_designated_places/2024/Wyoming_56/Wyoming_56_demographics.csv
[INFO] 2025-07-03 16:11:09 - Completed task for Wyoming (FIPS code 56)
Processing States: 100%|██████████| 12/12 [00:02<00:00,  4.32it/s]
[INFO] 2025-07-03 16:11:09 - All tasks have been processed. Check the output directory for the extracted state folders and demographic CSV files.
